# CosmicExplosions 2026 · BlackJAX tutorial

This is the learner notebook for
[CosmicExplosions 2026](https://cambridgetransients.github.io/CosmicExplosions2026/).
It follows one model through the whole workshop:

1. check the workshop environment and JAX random keys;
2. express a line-fitting model in Distrax and approximate it with Pathfinder;
3. run Nested Slice Sampling, then accelerate it with Pathfinder-informed
   posterior repartitioning.

Cells marked **TODO** are yours to complete. The fully worked reference is on
the `main` branch and at
[yallup.github.io/blackjax_tutorial](https://yallup.github.io/blackjax_tutorial/).

# Lesson 1 — Setup

This notebook assumes you have already opened the learner checkout and selected
its Python environment. Begin with the smoke test; there is no extra book setup
inside the notebook.

## Smoke test

In [ ]:
import blackjax
import distrax
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)

print("BlackJAX:", blackjax.__version__)
print("JAX device:", jax.devices()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)

A CPU device is exactly right for this tutorial.

JAX uses explicit random keys. Splitting the key makes every random draw
visible and reproducible:

In [ ]:
key = jax.random.key(7)
key, draw_key = jax.random.split(key)
draws = jax.random.normal(draw_key, shape=(5,))
draws

**Checkpoint:** the imports work, and you can explain why we split a random key
instead of relying on hidden global state.

# Lesson 2 — A Distrax model and Pathfinder

Inference is easier to change and reuse when the model and algorithm are
separate. We will build the best-practice structured model from the
[Nested Sampling Book line-fitting lesson](https://github.com/handley-lab/nested-sampling-book/blob/main/basic/line_fitting.ipynb),
then hand its log density to
[BlackJAX Pathfinder](https://blackjax-devs.github.io/sampling-book/algorithms/pathfinder.html).

## Make a noisy straight line

We will infer its slope, intercept, and noise.

In [ ]:
key = jax.random.key(17)
x = jnp.linspace(-2.0, 2.0, 40)
key, data_key = jax.random.split(key)
y = 0.8 * x - 0.2 + 0.25 * jax.random.normal(data_key, shape=x.shape)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(x, y, s=24, color="tab:blue", label="observations")
ax.set(xlabel="x", ylabel="y", title="A noisy straight line")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.show()

## Put the prior in one place

A dictionary is a JAX **PyTree**. Named parameters are harder to misuse than
positions such as `params[0]`. A Distrax `Joint` distribution owns both
sampling and `log_prob`, preventing those two definitions from drifting apart.

In [ ]:
# TODO 2.1
# Build a distrax.Joint with:
# slope      ~ Normal(0, 2)
# intercept  ~ Normal(0, 1)
# log_noise  ~ Normal(-1, 0.75)
prior = ...

Keep the likelihood readable and separate:

In [ ]:
def log_likelihood(params):
    # TODO 2.2
    # 1. Transform log_noise into a positive noise scale.
    # 2. Compute slope * x + intercept.
    # 3. Standardise the residuals.
    # 4. Return the pure-JAX Gaussian log likelihood
    #    -0.5 * sum(residual**2 + 2*log(noise) + log(2*pi)).
    raise NotImplementedError


def log_posterior(params):
    # TODO 2.3
    # Combine the prior log probability and log likelihood.
    raise NotImplementedError

Test one draw before compiling or fitting:

In [ ]:
key, init_key = jax.random.split(key)

# TODO 2.4: draw one initial position from the prior.
initial_position = ...

initial_position, log_posterior(initial_position)

The best-practice pattern is now visible:

1. semantic names in a PyTree;
2. one distribution object for prior sampling and density evaluation;
3. small, pure likelihood and posterior functions;
4. one cheap check before inference.

## PPLs and the BlackJAX philosophy

Distrax is a distribution and bijector library rather than a complete
probabilistic programming language (PPL). It suits this small example because
an explicit likelihood keeps the assumptions visible.
[TensorFlow Probability](https://www.tensorflow.org/probability) is another
model-building option, with a larger collection of distributions, bijectors,
joint distributions, and inference tools, including a JAX substrate.

Both libraries can compose named distributions: Distrax provides `Joint`,
which we use above, and TFP provides `JointDistribution`. They bundle sampling
and joint-density evaluation, but neither object is itself a PPL.

Full PPLs such as NumPyro, PyMC, and Stan go further: they provide a language
for describing a generative model and typically manage parameter transforms,
conditioning, latent-variable names, and construction of the joint log
density.

| Layer | Main question | Examples |
| --- | --- | --- |
| Distribution toolkit | How do I sample from, score, and compose distributions? | Distrax (`Joint`), TensorFlow Probability (`JointDistribution`) |
| Probabilistic programming | How do I express and manage a generative model? | NumPyro, PyMC, Stan |
| Inference algorithms | How does state move through this target density? | BlackJAX Pathfinder, NUTS, SMC, NSS |

BlackJAX is an **inference library, not a modelling language**. Its
[design philosophy](https://blackjax-devs.github.io/blackjax/developer/design_principles.html)
favours small pure functions, explicit random keys and state, PyTrees, and
`init`/`step` interfaces. We do a little more visible plumbing, but the model is
not owned by one sampler and the algorithmic pieces remain inspectable and
recomposable.

The layers are complementary. A larger project can define its model in a PPL,
extract a JAX-compatible log density, and give that function to BlackJAX. Its
documentation includes integrations for NumPyro, PyMC, and TensorFlow
Probability. We use Distrax today to make the boundary especially clear; TFP
is an equally valid model-building route, but it is not required for the
workshop.

## Compose the model with Pathfinder

[Pathfinder](https://arxiv.org/abs/2108.03782) follows an L-BFGS optimisation
path and builds local Gaussian approximations from inverse-Hessian estimates.
BlackJAX separates fitting from cheap sampling.

In [ ]:
# TODO 2.5: compose Pathfinder with log_posterior.
pathfinder = ...

key, fit_key, sample_key = jax.random.split(key, 3)

# TODO 2.6: initialise Pathfinder at initial_position with ftol=1e-5.
pathfinder_state, pathfinder_info = ...

# TODO 2.7: draw 2,000 samples from the fitted approximation.
pathfinder_samples, _ = ...

Inspect the approximation:

In [ ]:
for name in ("slope", "intercept", "log_noise"):
    values = pathfinder_samples[name]
    print(
        f"{name:>10}: "
        f"mean={values.mean(): .3f}, "
        f"sd={values.std(): .3f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].scatter(x, y, s=18, color="tab:blue", alpha=0.75)
for i in range(0, 2_000, 40):
    line = (
        pathfinder_samples["slope"][i] * x
        + pathfinder_samples["intercept"][i]
    )
    axes[0].plot(x, line, color="tab:orange", alpha=0.08)
axes[0].set(xlabel="x", ylabel="y", title="Posterior lines")

axes[1].scatter(
    pathfinder_samples["slope"],
    pathfinder_samples["intercept"],
    s=8,
    alpha=0.2,
    color="tab:green",
)
axes[1].set(
    xlabel="slope",
    ylabel="intercept",
    title="Pathfinder draws",
)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

**Checkpoint:** you can point to where the prior lives, where the likelihood
lives, and exactly what Pathfinder receives.

# Lesson 3 — NSS and posterior repartitioning

We now reuse the same prior, likelihood, and Pathfinder approximation.
[Nested Slice Sampling](https://arxiv.org/abs/2601.23252) gives us posterior
draws and the Bayesian evidence. Then we turn Pathfinder into a proposal prior
and use posterior repartitioning to reduce the amount of compression.

## What nested sampling adds

The evidence is the average likelihood under the prior:

$$
\mathcal{Z}
=
\int \mathcal{L}(\theta)\,\pi(\theta)\,\mathrm{d}\theta.
$$

Nested sampling keeps a population of **live points**. It repeatedly records
the lowest-likelihood points as **dead points**, then replaces them with points
above the new likelihood threshold.

- `num_live` controls resolution;
- `num_inner_steps` controls reliability;
- `num_delete` controls parallelism.

For NSS, start with at least `max(5, 2 * dimension)` inner steps and test that
increasing it does not change the answer.

In [ ]:
num_dimensions = 3
num_live = 150
num_delete = 50

# TODO 3.1: choose at least max(5, 2 * num_dimensions).
num_inner_steps = ...

## Write one reusable NSS loop

The loop stops when the evidence still contained in the live points is below
about five per cent of the accumulated dead-point evidence.

In [ ]:
def run_nss(key, logprior, loglike, initial_particles, max_batches=200):
    # TODO 3.2: create blackjax.nss from the supplied logprior and loglike.
    sampler = ...

    key, init_key = jax.random.split(key)
    live = sampler.init(initial_particles, init_key)
    step = jax.jit(sampler.step)
    dead = []

    for batch in range(max_batches):
        # TODO 3.3:
        # Stop when logZ_live - logZ < -3.
        if ...:
            break

        key, step_key = jax.random.split(key)
        live, info = step(step_key, live)
        dead.append(info)
    else:
        raise RuntimeError("NSS did not reach the stopping rule")

    complete_run = blackjax.ns.utils.finalise(live, dead)
    return key, complete_run, len(dead)

Initial live points must be drawn from the same prior whose `log_prob` NSS
evaluates:

In [ ]:
key, prior_key, nss_key = jax.random.split(key, 3)

# TODO 3.4: draw num_live particles from prior.
initial_live = ...

key, direct_run, direct_batches = run_nss(
    nss_key,
    prior.log_prob,
    log_likelihood,
    initial_live,
)

print(
    f"direct NSS: {direct_batches} batches, "
    f"{direct_batches * num_delete} dead points"
)

The final live points are not posterior draws. Finalisation joins them to the
dead-point history, after which `blackjax.ns.utils.sample` applies the correct
nested-sampling weights.

In [ ]:
key, direct_z_key, direct_sample_key = jax.random.split(key, 3)

# TODO 3.5:
# Use blackjax.ns.utils.log_weights with shape=128, then logsumexp
# over the particle axis (axis=0).
direct_logz = ...

# TODO 3.6:
# Resample 2,000 equally weighted particles and keep their .position.
direct_samples = ...

print(
    f"direct NSS log evidence: "
    f"{direct_logz.mean():.3f} ± {direct_logz.std():.3f}"
)

## Turn Pathfinder into a proposal

Turn the marginal means and standard deviations of the Pathfinder draws into a
normalised, factorised `distrax.Joint` proposal. Doubling each scale makes the
proposal enclose rather than tightly clip the approximation.

In [ ]:
parameter_names = ("slope", "intercept", "log_noise")
pathfinder_proposal = distrax.Joint(
    {
        name: distrax.Normal(
            loc=pathfinder_samples[name].mean(),
            scale=2.0 * pathfinder_samples[name].std() + 1e-6,
        )
        for name in parameter_names
    }
)

> **Why factorise?** `distrax.Joint` samples and scores the same named PyTree as
> the model, so there is no packing or unpacking step. This discards Pathfinder's
> estimated correlations and can be less efficient on strongly correlated
> problems. Posterior repartitioning remains exact because the density-ratio
> correction accounts for the difference; only efficiency changes. A dense
> multivariate proposal is a useful later optimisation when the extra vector
> conversion is justified.

## Repartition without changing the answer

Posterior repartitioning replaces the original prior $\pi$ with a proposal
$\tilde\pi$ and corrects the likelihood:

$$
\tilde{\mathcal{L}}(\theta)
=
\mathcal{L}(\theta)
\frac{\pi(\theta)}{\tilde\pi(\theta)}.
$$

The product, and therefore the posterior and evidence, stays unchanged:

$$
\tilde{\mathcal{L}}(\theta)\tilde\pi(\theta)
=
\mathcal{L}(\theta)\pi(\theta).
$$

Following the support-safe
[SuperNest](https://arxiv.org/abs/2212.01760) idea, use a mixture of 90%
Pathfinder proposal and 10% original prior.

In [ ]:
pathfinder_weight = 0.9


def repartitioned_logprior(params):
    # TODO 3.7:
    # Use jnp.logaddexp to evaluate the two-component mixture density.
    raise NotImplementedError


def repartitioned_loglikelihood(params):
    # TODO 3.8:
    # log L_tilde = log L + log pi - log pi_tilde
    raise NotImplementedError

Test the invariant before sampling:

In [ ]:
key, check_key = jax.random.split(key)
check_position = prior.sample(seed=check_key)

original_joint = log_posterior(check_position)
repartitioned_joint = (
    repartitioned_logprior(check_position)
    + repartitioned_loglikelihood(check_position)
)

print("log-joint difference:", repartitioned_joint - original_joint)

The new live points must come from the new mixture prior. Make the split
literal: 90% Pathfinder draws and 10% original-prior draws.

In [ ]:
num_pathfinder = round(pathfinder_weight * num_live)
num_original = num_live - num_pathfinder

key, pf_prior_key, original_prior_key = jax.random.split(key, 3)
pathfinder_particles = pathfinder_proposal.sample(
    seed=pf_prior_key,
    sample_shape=(num_pathfinder,),
)
original_particles = prior.sample(
    seed=original_prior_key,
    sample_shape=(num_original,),
)

# TODO 3.9: Concatenate each matching leaf of these two PyTrees.
initial_repartitioned = ...

> **A simple, stratified mixture.** With 50 live points this produces 45
> Pathfinder draws and 5 original-prior draws. The fixed allocation makes the
> idea easy to see and avoids sampling particles that are then discarded. It is
> stratified rather than a strictly independent draw from the mixture; for the
> latter, draw a fresh Bernoulli component label for every particle. The
> mixture density and repartitioning correction stay the same.

Compose the same NSS loop with the new functions:

In [ ]:
key, repartitioned_key = jax.random.split(key)

key, repartitioned_run, repartitioned_batches = run_nss(
    repartitioned_key,
    repartitioned_logprior,
    repartitioned_loglikelihood,
    initial_repartitioned,
)

## Compare the two exact runs

In [ ]:
key, repartitioned_z_key, repartitioned_sample_key = jax.random.split(
    key, 3
)

repartitioned_logz = jax.scipy.special.logsumexp(
    blackjax.ns.utils.log_weights(
        repartitioned_z_key,
        repartitioned_run,
        shape=128,
    ),
    axis=0,
)
repartitioned_samples = blackjax.ns.utils.sample(
    repartitioned_sample_key,
    repartitioned_run,
    shape=2_000,
).position

print(f"{'run':>18}  {'log evidence':>18}  {'batches':>8}")
print(
    f"{'direct':>18}  "
    f"{direct_logz.mean():>8.3f} ± {direct_logz.std():.3f}  "
    f"{direct_batches:>8}"
)
print(
    f"{'repartitioned':>18}  "
    f"{repartitioned_logz.mean():>8.3f} "
    f"± {repartitioned_logz.std():.3f}  "
    f"{repartitioned_batches:>8}"
)
print(
    f"\nrun-length reduction: "
    f"{direct_batches / repartitioned_batches:.1f}×"
)

The two evidence estimates should agree within their uncertainties.
Repartitioning should need fewer batches and usually has a smaller evidence
uncertainty because Pathfinder removed much of the compression.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].scatter(
    direct_samples["slope"],
    direct_samples["intercept"],
    s=8,
    alpha=0.16,
    label="direct NSS",
)
axes[0].scatter(
    repartitioned_samples["slope"],
    repartitioned_samples["intercept"],
    s=8,
    alpha=0.16,
    label="repartitioned NSS",
)
axes[0].scatter(
    pathfinder_samples["slope"][::8],
    pathfinder_samples["intercept"][::8],
    s=8,
    alpha=0.16,
    label="Pathfinder",
)
axes[0].set(
    xlabel="slope",
    ylabel="intercept",
    title="Three routes to the posterior",
)
axes[0].legend(frameon=False, fontsize=8)

axes[1].scatter(
    x,
    y,
    s=18,
    color="tab:blue",
    alpha=0.75,
    label="observations",
)
for i in range(0, 2_000, 40):
    line = (
        repartitioned_samples["slope"][i] * x
        + repartitioned_samples["intercept"][i]
    )
    axes[1].plot(x, line, color="tab:orange", alpha=0.08)
axes[1].set(
    xlabel="x",
    ylabel="y",
    title="Repartitioned NSS posterior lines",
)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Final checkpoint

You should now be able to explain:

1. why the model uses named PyTrees and a single prior object;
2. why Pathfinder only needs `log_posterior`;
3. why NSS needs `logprior` and `loglike` separately;
4. why final live points are not posterior samples;
5. why the repartitioning density ratio leaves the answer unchanged.

Further reading:

- [BlackJAX Sampling Book](https://blackjax-devs.github.io/sampling-book/)
- [Nested Sampling Book](https://handley-lab.co.uk/nested-sampling-book/)
- [Nested Slice Sampling](https://arxiv.org/abs/2601.23252)
- [Bayesian posterior repartitioning](https://arxiv.org/abs/1908.04655)
- [Nested Sampling with Slice-within-Gibbs](https://arxiv.org/abs/2602.17414)
- [SwiG research implementation](https://github.com/yallup/swig)